# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shoriful-mynul/flyrank-assignment1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

### Task Type: Ranking

I frame the Content Refresh Prioritization lane as a **ranking task**.

The decision is not simply whether a page is good or bad. The practical question is: **which existing content pages should be reviewed and refreshed first?**

The model would assign each page a priority score, and the content team could use that score to rank pages from higher to lower refresh priority.

Ranking fits this problem because the team has limited time and needs to focus on the highest-value opportunities first. The goal is therefore to produce a useful ordered list rather than only a yes/no prediction.

The ranking would use observable content, traffic, engagement, freshness, and search-performance signals available before the decision point.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import pandas as pd

repo_path = "/content/flyrank-ml-internship"

if not os.path.exists(repo_path):
    !git clone https://github.com/shoriful-mynul/flyrank-ml-internship.git

%cd /content/flyrank-ml-internship

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Task type: Ranking")
print("Rows available:", len(df))
print("Unique content items:", df["content_id"].nunique())


/content/flyrank-ml-internship
Task type: Ranking
Rows available: 30000
Unique content items: 30000


## 2. Target or proxy

### Provisional proxy: observed decline status

For this starter exercise, I will use `is_declining_label` as a **proxy** for refresh priority, where:

- `1` means the page is currently labelled as declining.
- `0` means it is not labelled as declining.

This proxy is derived from the current `trend_direction` field, so it is not the ideal target for a future-looking production model. It is useful here as a simple starting point for framing and checking whether the ranking idea is measurable with the starter data.

A stronger future version of this project would define the target from a later time window, for example whether a page experiences a measurable decline during the next 30 days after the decision point.

For the eventual model, current trend fields used to construct this proxy would not be used as input features, because that would create leakage or make the model simply reproduce the existing label.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Proxy column: is_declining_label")
print(df["is_declining_label"].value_counts().sort_index())
print("\nProxy rate:", round(df["is_declining_label"].mean() * 100, 2), "%")


Proxy column: is_declining_label
is_declining_label
0    13738
1    16262
Name: count, dtype: int64

Proxy rate: 54.21 %


## 3. Success metric

### Metric: Precision@K

I will use **Precision@K** as the main success metric, with an initial focus on **Precision@50**.

This metric matches the real decision because a content team has limited time and will review only the highest-priority pages first.

Precision@50 asks: among the 50 pages placed at the top of the ranked list, how many are actually positive according to the chosen proxy?

A higher Precision@50 would mean that the limited review capacity is being concentrated on pages that match the observed refresh-relevant signal more often.

This metric does not prove that refreshing those pages will increase traffic. It only measures how well the ranking concentrates the selected proxy signal near the top of the queue.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

K = 50

positive_count = df["is_declining_label"].sum()
base_rate = df["is_declining_label"].mean()

print("Evaluation metric: Precision@50")
print("Top-K:", K)
print("Positive proxy rate:", round(base_rate * 100, 2), "%")
print("Expected positives in a random top-50 sample:", round(base_rate * K, 2))


Evaluation metric: Precision@50
Top-K: 50
Positive proxy rate: 54.21 %
Expected positives in a random top-50 sample: 27.1


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one existing content item (page)**.

Each row represents one pseudonymized content item for one client. The starter dataset contains 30,000 content records and 32 clients.

For this lane, the decision is made at the content-item level because the content manager needs to decide which individual pages should be reviewed and potentially refreshed.

The dataframe below shows the page-level fields that are relevant to the framing: content type, search demand, traffic, freshness, engagement, and the provisional decline proxy.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

lane_slice = df[
    [
        "content_id",
        "client_id",
        "content_type",
        "main_intent",
        "search_volume",
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "content_age_days",
        "days_since_last_update",
        "ctr",
        "avg_position",
        "engagement_rate",
        "is_declining_label",
    ]
].copy()

print("Unit of analysis: one content item (page) per row")
print("Dataframe shape:", lane_slice.shape)

display(lane_slice.head(10))


Unit of analysis: one content item (page) per row
Dataframe shape: (30000, 14)


,content_id,client_id,content_type,main_intent,search_volume,impressions_90d,clicks_90d,sessions_90d,content_age_days,days_since_last_update,ctr,avg_position,engagement_rate,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,10.0,3803,29,17,187,20,0.76,10.6,5.88,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,90.0,15320,7,9,445,25,0.05,20.3,0.00,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,0.0,12581,11,11,141,20,0.09,36.5,0.00,1
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,10.0,11751,58,78,463,22,0.49,6.2,1.28,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,0.0,19140,24,145,263,14,0.13,44.0,0.00,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,transactional,720.0,3970,1,5,147,20,0.03,8.5,0.00,1
6,content_9a34b442b552,client_8722616204,keyword article,informational,0.0,20,0,1,90,20,0.00,7.0,0.00,1
7,content_a63219c6e95a,client_19581e27de,keyword article,commercial,590.0,1724,1,28,445,22,0.06,21.2,3.57,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,informational,0.0,32574,29,68,90,20,0.09,46.0,5.88,1
9,content_c27558df2b0c,client_19581e27de,keyword article,informational,0.0,1240,2,3,257,104,0.16,4.9,0.00,1


## 5. Why ML beats a fixed rule here


A simple fixed rule could rank pages using one threshold, such as refreshing every page that has not been updated for more than a certain number of days. However, refresh priority is unlikely to depend on one signal alone.

A page can have different combinations of search demand, traffic, CTR, position, engagement, content age, and freshness. These signals can interact in ways that are difficult to capture with a small set of hand-written thresholds.

ML is therefore worth testing because it can combine multiple observable signals and learn patterns from historical outcomes or proxies instead of relying on one fixed cutoff.

The purpose is not to replace editorial judgment. The output would be a ranked decision-support queue that helps a content manager decide which pages deserve attention first.

A fixed rule remains an important baseline. ML should only earn its place if it produces a more useful ranking on held-out data.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

candidate_signals = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
]

print("Candidate observable signals for future ranking:")
for signal in candidate_signals:
    print("-", signal)

print("\nNumber of candidate signals:", len(candidate_signals))


Candidate observable signals for future ranking:
- search_volume
- impressions_90d
- clicks_90d
- sessions_90d
- content_age_days
- days_since_last_update
- ctr
- avg_position
- engagement_rate

Number of candidate signals: 9


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] Run the notebook top to bottom with Runtime → Run all.
- [x] Save the executed notebook back to the GitHub repository.
- [x] Confirm that no client names, private URLs, queries, or sensitive information are included.